# Spam Classification with RNN & CNN

In [1]:
import os
import re
import pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

## 1. Load Dataset

In [2]:
data = []
with open("spam.txt", "r", encoding="utf-8") as f:
    for line in f:
        if "\t" in line:
            text, label = line.strip().split("\t")
            data.append((text, int(label)))

df = pd.DataFrame(data, columns=["text", "label"])
print("Dataset size:", len(df))
print(df.head())

Dataset size: 1547
                                                text  label
0  Go until jurong point, crazy.. Available only ...      0
1                      Ok lar... Joking wif u oni...      0
2  U dun say so early hor... U c already then say...      0
3  Nah I don't think he goes to usf, he lives aro...      0
4  Even my brother is not like to speak with me. ...      0


## 2. Preprocessing

In [3]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)
    return text

df["clean_text"] = df["text"].apply(clean_text)

# Load vocab (dict with vocab, word2idx, idx2word)
with open("checkpoints/vocab.pkl", "rb") as f:
    vocab_data = pickle.load(f)

vocab = vocab_data["vocab"]
word2idx = vocab_data["word2idx"]
idx2word = vocab_data["idx2word"]

PAD_IDX = word2idx.get("<pad>", 0)
UNK_IDX = word2idx.get("<unk>", PAD_IDX)

def encode_sentence(sentence, max_len=50):
    tokens = sentence.split()
    ids = [word2idx.get(t, UNK_IDX) for t in tokens]
    if len(ids) < max_len:
        ids += [PAD_IDX] * (max_len - len(ids))
    else:
        ids = ids[:max_len]
    return ids

MAX_LEN = 50
X = np.array([encode_sentence(s, MAX_LEN) for s in df["clean_text"]])
y = df["label"].values

## 3. Load Pretrained Embeddings

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class SkipGramNegSampling(nn.Module):
    def __init__(self, vocab_size, embedding_dim, svd_tensor):
        super().__init__()
        self.in_embeddings = nn.Embedding.from_pretrained(svd_tensor.clone().float(), freeze=False, padding_idx=PAD_IDX)
        self.out_embeddings = nn.Embedding(vocab_size, embedding_dim)
        nn.init.uniform_(self.out_embeddings.weight, -0.5/embedding_dim, 0.5/embedding_dim)

    def forward(self, center_words, pos_context_words, neg_context_words):
        c = self.in_embeddings(center_words)                     # (batch, emb)
        pos = self.out_embeddings(pos_context_words)             # (batch, emb)
        neg = self.out_embeddings(neg_context_words)             # (batch, K, emb)

        pos_score = torch.sum(c * pos, dim=1)                    # (batch,)
        pos_loss = torch.log(torch.sigmoid(pos_score) + 1e-10)

        neg_score = torch.bmm(neg, c.unsqueeze(2)).squeeze(2)    # (batch, K)
        neg_loss = torch.log(torch.sigmoid(-neg_score) + 1e-10).sum(1)

        loss = -(pos_loss + neg_loss).mean()
        return loss

def find_last_checkpoint(directory, prefix="checkpoint_"):
    if not os.path.isdir(directory):
        return None
    checkpoints = [f for f in os.listdir(directory) if f.endswith(".pt") and prefix in f]
    if not checkpoints:
        return None
    epochs = []
    for f in checkpoints:
        try:
            num = int(f.replace(prefix, "").replace(".pt", ""))
            epochs.append((num, f))
        except:
            continue
    if not epochs:
        return None
    latest = max(epochs, key=lambda x: x[0])[1]
    return os.path.join(directory, latest)

ckpt_path = find_last_checkpoint("checkpoints")
if ckpt_path is None:
    raise FileNotFoundError("No checkpoint found in checkpoints/")

checkpoint = torch.load(ckpt_path, map_location=device)
embedding_dim = checkpoint["model_state_dict"]["in_embeddings.weight"].shape[1]
vocab_size = checkpoint["model_state_dict"]["in_embeddings.weight"].shape[0]

sg_model = SkipGramNegSampling(vocab_size, embedding_dim, torch.rand(vocab_size, embedding_dim)).to(device)
sg_model.load_state_dict(checkpoint["model_state_dict"])

pretrained_weights = sg_model.in_embeddings.weight.detach().float().cpu()
print("Loaded pretrained embeddings:", pretrained_weights.shape)

Loaded pretrained embeddings: torch.Size([27206, 300])


## 4. Dataset & DataLoader

In [5]:
class SMSDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

## 5. RNN Classifier

In [6]:
class RNNClassifier(nn.Module):
    def __init__(self, pretrained_weights, hidden_dim=128, num_layers=1, num_classes=2):
        super(RNNClassifier, self).__init__()
        vocab_size, embedding_dim = pretrained_weights.shape
        self.embedding = nn.Embedding.from_pretrained(pretrained_weights, freeze=False, padding_idx=PAD_IDX)
        self.rnn = nn.LSTM(embedding_dim, hidden_dim, num_layers,
                           batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        out, (hn, cn) = self.rnn(x)
        if self.rnn.bidirectional:
            h_forward = hn[-2, :, :]
            h_backward = hn[-1, :, :]
            h = torch.cat((h_forward, h_backward), dim=1)
        else:
            h = hn[-1]
        return self.fc(h)

## 6. CNN Classifier

In [7]:
class CNNClassifier(nn.Module):
    def __init__(self, pretrained_weights, num_classes=2, num_filters=100, filter_sizes=(3,4,5)):
        super(CNNClassifier, self).__init__()
        vocab_size, embedding_dim = pretrained_weights.shape
        self.embedding = nn.Embedding.from_pretrained(pretrained_weights, freeze=False, padding_idx=PAD_IDX)
        self.convs = nn.ModuleList([
            nn.Conv2d(1, num_filters, (fs, embedding_dim)) for fs in filter_sizes
        ])
        self.fc = nn.Linear(num_filters * len(filter_sizes), num_classes)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.embedding(x)                   # (batch, seq, emb)
        x = x.unsqueeze(1)                      # (batch, 1, seq, emb)
        conv_outs = []
        for conv in self.convs:
            c = torch.relu(conv(x))             # (batch, num_filters, seq-fs+1, 1)
            c = c.squeeze(3)                    # (batch, num_filters, seq-fs+1)
            c = torch.max(c, dim=2)[0]          # (batch, num_filters)
            conv_outs.append(c)
        x = torch.cat(conv_outs, dim=1)
        x = self.dropout(x)
        return self.fc(x)

## 7. Training & Evaluation Utils

In [8]:
def train_eval(model_class, pretrained_weights, X, y, folds=5, epochs=5, batch_size=64):
    kf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=42)
    metrics = {"acc": [], "prec": [], "rec": [], "f1": []}

    for fold, (train_idx, test_idx) in enumerate(kf.split(X, y)):
        print(f"\n===== Fold {fold+1} =====")
        train_ds = SMSDataset(X[train_idx], y[train_idx])
        test_ds = SMSDataset(X[test_idx], y[test_idx])
        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
        test_loader = DataLoader(test_ds, batch_size=batch_size)

        model = model_class(pretrained_weights).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=0.001)

        model.train()
        for epoch in range(epochs):
            total_loss = 0.0
            for Xb, yb in train_loader:
                Xb, yb = Xb.to(device), yb.to(device)
                optimizer.zero_grad()
                preds = model(Xb)
                loss = criterion(preds, yb)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5)
                optimizer.step()
                total_loss += loss.item()
            print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

        model.eval()
        y_true, y_pred = [], []
        with torch.no_grad():
            for Xb, yb in test_loader:
                Xb = Xb.to(device)
                preds = model(Xb).argmax(dim=1).cpu().numpy()
                y_true.extend(yb.numpy())
                y_pred.extend(preds)

        acc = accuracy_score(y_true, y_pred)
        prec = precision_score(y_true, y_pred, zero_division=0)
        rec = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)

        metrics["acc"].append(acc)
        metrics["prec"].append(prec)
        metrics["rec"].append(rec)
        metrics["f1"].append(f1)
        print(f"Fold {fold+1} → Acc: {acc:.4f}, Prec: {prec:.4f}, Rec: {rec:.4f}, F1: {f1:.4f}")

    for k in metrics:
        print(f"{k.upper()}: {np.mean(metrics[k]):.4f} ± {np.std(metrics[k]):.4f}")
    return metrics

## 8. Run Experiments

In [9]:
print("\n===== RNN Model Results =====")
rnn_metrics = train_eval(RNNClassifier, pretrained_weights, X, y)

print("\n===== CNN Model Results =====")
cnn_metrics = train_eval(CNNClassifier, pretrained_weights, X, y)


===== RNN Model Results =====

===== Fold 1 =====
Epoch 1, Loss: 0.6201
Epoch 2, Loss: 0.2941
Epoch 3, Loss: 0.1888
Epoch 4, Loss: 0.0887
Epoch 5, Loss: 0.0481
Fold 1 → Acc: 0.9065, Prec: 0.8580, Rec: 0.9667, F1: 0.9091

===== Fold 2 =====
Epoch 1, Loss: 0.6478
Epoch 2, Loss: 0.3864
Epoch 3, Loss: 0.1932
Epoch 4, Loss: 0.0853
Epoch 5, Loss: 0.0487
Fold 2 → Acc: 0.9355, Prec: 0.9710, Rec: 0.8933, F1: 0.9306

===== Fold 3 =====
Epoch 1, Loss: 0.6636
Epoch 2, Loss: 0.4161
Epoch 3, Loss: 0.1863
Epoch 4, Loss: 0.0968
Epoch 5, Loss: 0.0460
Fold 3 → Acc: 0.9450, Prec: 0.9400, Rec: 0.9463, F1: 0.9431

===== Fold 4 =====
Epoch 1, Loss: 0.5991
Epoch 2, Loss: 0.3050
Epoch 3, Loss: 0.1628
Epoch 4, Loss: 0.0826
Epoch 5, Loss: 0.0397
Fold 4 → Acc: 0.9515, Prec: 0.9653, Rec: 0.9329, F1: 0.9488

===== Fold 5 =====
Epoch 1, Loss: 0.6188
Epoch 2, Loss: 0.2724
Epoch 3, Loss: 0.1211
Epoch 4, Loss: 0.0672
Epoch 5, Loss: 0.0389
Fold 5 → Acc: 0.9450, Prec: 0.9853, Rec: 0.8993, F1: 0.9404
ACC: 0.9367 ± 0.015